<a href="https://colab.research.google.com/github/charansai12322/tradingagent/blob/main/tradingagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key = userdata.get('sec1')

model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a stock trading analyst."
)

In [ ]:
import requests
from google.colab import userdata

twelvedata_api_key = userdata.get('TWELVEDATA_API_KEY')

url = "https://api.twelvedata.com/price"
params = {
    "symbol": "AAPL",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'price': '337.39'}


In [ ]:
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current market price for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/price"
    params = {"symbol": ticker, "apikey": twelvedata_api_key}
    response = requests.get(url, params=params)
    data = response.json()
    if "price" not in data:
        return f"Could not fetch price for {ticker}: {data.get('message', 'Unknown error')}"
    return f"{ticker} current price: ${float(data['price']):.2f}"

In [ ]:
print(get_stock_price.invoke({"ticker": "AAPL"}))

AAPL current price: $337.50


In [ ]:
url = "https://api.twelvedata.com/time_series"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "outputsize": "10",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock'}, 'values': [{'datetime': '2026-10-08', 'open': '336.83', 'high': '338.225', 'low': '335.91', 'close': '337.59', 'volume': '214863'}, {'datetime': '2026-10-07', 'open': '336.95999', 'high': '338.67001', 'low': '332.78000', 'close': '336.67001', 'volume': '34128200'}, {'datetime': '2026-10-06', 'open': '332.28000', 'high': '334.38000', 'low': '330.62000', 'close': '333.63000', 'volume': '30449000'}, {'datetime': '2026-10-05', 'open': '332.82001', 'high': '336.20999', 'low': '331.64999', 'close': '332.89001', 'volume': '34400900'}, {'datetime': '2026-10-02', 'open': '333.26001', 'high': '334.54001', 'low': '330.60999', 'close': '333.69000', 'volume': '33278600'}, {'datetime': '2026-10-01', 'open': '330', 'high': '332.48001', 'low': '325.81000', 'close': '330.32001', 'volume': '36306300'}, {'datetime': '2026-09-30', 'o

In [ ]:
@tool
def get_price_history(ticker: str) -> str:
    """Get the last 10 days of closing prices for a stock. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/time_series"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "outputsize": "10",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch price history for {ticker}: {data.get('message', 'Unknown error')}"
    prices = [
        f"  {item['datetime']}: ${float(item['close']):.2f}"
        for item in reversed(data["values"])
    ]
    return f"{ticker} closing prices (last 10 days):\n" + "\n".join(prices)

In [ ]:
url = "https://api.twelvedata.com/rsi"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "14",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'RSI - Relative Strength Index', 'series_type': 'close', 'time_period': 14}}, 'values': [{'datetime': '2026-10-08', 'rsi': '58.51575'}, {'datetime': '2026-10-07', 'rsi': '57.67847'}, {'datetime': '2026-10-06', 'rsi': '54.55918'}, {'datetime': '2026-10-05', 'rsi': '53.78934'}, {'datetime': '2026-10-02', 'rsi': '54.71997'}, {'datetime': '2026-10-01', 'rsi': '51.43312'}, {'datetime': '2026-09-30', 'rsi': '54.36923'}, {'datetime': '2026-09-29', 'rsi': '50.87808'}, {'datetime': '2026-09-28', 'rsi': '61.79236'}, {'datetime': '2026-09-25', 'rsi': '65.67332'}, {'datetime': '2026-09-24', 'rsi': '61.32250'}, {'datetime': '2026-09-23', 'rsi': '62.90377'}, {'datetime': '2026-09-22', 'rsi': '66.87815'}, {'datetime': '2026-09-21', 'rsi': '66.32085'}, {'datetime': '2026-09-18', 'rsi': '64.25365'}, {'datetime

In [ ]:
@tool
def calculate_rsi(ticker: str) -> str:
    """Calculate the current RSI for a stock to check if it is overbought or oversold. Example: ticker='AAPL'"""
    url = "https://api.twelvedata.com/rsi"
    params = {
        "symbol": ticker,
        "interval": "1day",
        "time_period": "14",
        "apikey": twelvedata_api_key
    }
    response = requests.get(url, params=params)
    data = response.json()
    if "values" not in data:
        return f"Could not fetch RSI for {ticker}: {data.get('message', 'Unknown error')}"
    rsi = float(data["values"][0]["rsi"])
    if rsi > 70:
        signal = "Overbought — price may drop soon"
    elif rsi < 30:
        signal = "Oversold — price may rise soon"
    else:
        signal = "Normal"
    return f"{ticker} RSI(14): {rsi:.1f} → {signal}"

In [ ]:
url = "https://api.twelvedata.com/sma"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "time_period": "50",
    "apikey": twelvedata_api_key
}

response = requests.get(url, params=params)
print(response.json())

{'meta': {'symbol': 'AAPL', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'SMA - Simple Moving Average', 'series_type': 'close', 'time_period': 50}}, 'values': [{'datetime': '2026-10-08', 'sma': '322.26600'}, {'datetime': '2026-10-07', 'sma': '322.27980'}, {'datetime': '2026-10-06', 'sma': '322.34800'}, {'datetime': '2026-10-05', 'sma': '322.41360'}, {'datetime': '2026-10-02', 'sma': '322.41620'}, {'datetime': '2026-10-01', 'sma': '322.17560'}, {'datetime': '2026-09-30', 'sma': '322.087000'}, {'datetime': '2026-09-29', 'sma': '321.98140'}, {'datetime': '2026-09-28', 'sma': '321.92520'}, {'datetime': '2026-09-25', 'sma': '321.83200'}, {'datetime': '2026-09-24', 'sma': '321.67580'}, {'datetime': '2026-09-23', 'sma': '321.50740'}, {'datetime': '2026-09-22', 'sma': '321.064199'}, {'datetime': '2026-09-21', 'sma': '320.61540'}, {'datetime': '2026-09-18', 'sma': '320.1422

In [ ]:
@tool
def get_sma(ticker: str) -> str:
    """Get the 50-day and 200-day Simple Moving Averages to check the long-term price trend. Example: ticker='AAPL'"""
    base_params = {
        "symbol": ticker,
        "interval": "1day",
        "apikey": twelvedata_api_key
    }

    response_50 = requests.get(
        "https://api.twelvedata.com/sma",
        params={**base_params, "time_period": "50"}
    )
    response_200 = requests.get(
        "https://api.twelvedata.com/sma",
        params={**base_params, "time_period": "200"}
    )

    data_50 = response_50.json()
    data_200 = response_200.json()

    if "values" not in data_50 or "values" not in data_200:
        return f"Could not fetch SMA for {ticker}"

    sma_50 = float(data_50["values"][0]["sma"])
    sma_200 = float(data_200["values"][0]["sma"])

    if sma_50 > sma_200:
        trend = "Golden Cross — long-term uptrend (Bullish)"
    else:
        trend = "Death Cross — long-term downtrend (Bearish)"

    return f"{ticker} 50-day SMA: ${sma_50:.2f} | 200-day SMA: ${sma_200:.2f} | {trend}"

In [ ]:
system_prompt = """You are a stock trading analyst.

When asked to analyse a stock, follow these steps in order:
1. Get the current price → get_stock_price
2. Get the recent price history → get_price_history
3. Calculate RSI → calculate_rsi
4. Get the 50-day and 200-day SMA → get_sma

Then output your analysis in this format:

DECISION: BUY / SELL / HOLD
CONFIDENCE: High / Medium / Low

Current Price: $[price]
Recent trend: [describe the direction from price history]

RSI(14): [value] → [Overbought / Neutral / Oversold]
SMA: [Golden Cross / Death Cross] — [short explanation]

Rationale: [2 sentences explaining the decision based on the signals above]

Do not skip any tool. Always run all four before giving a decision."""

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_stock_price, get_price_history, calculate_rsi, get_sma],
    system_prompt=system_prompt
)

In [ ]:
!pip install finnhub-python

In [ ]:
import finnhub
from google.colab import userdata

finnhub_client = finnhub.Client(api_key=userdata.get('FINNHUB_API_KEY'))

In [ ]:
from datetime import datetime, timedelta

today = datetime.now().strftime("%Y-%m-%d")
week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

news = finnhub_client.company_news("AAPL", _from=week_ago, to=today)
print(news[:2])

[{'category': 'company', 'datetime': 1791467060, 'headline': 'Apple’s Tim Cook reveals one of his major new projects: ‘You want to give part of yourself’', 'id': 142727094, 'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png', 'related': 'AAPL', 'source': 'Yahoo', 'summary': 'King Charles and Elton John part of work to ‘focus on young people, young curious people’, Tim Cook tells David Phelan in the former CEO’s first interview since he stepped down', 'url': 'https://finnhub.io/api/news?id=001c42a417a33e4df1daac6082f5f294b6fcca34aabe8081095d4bef94b783b2'}, {'category': 'company', 'datetime': 1791460813, 'headline': 'AI Smart Glasses Will Be Huge. But Meta Faces A Problem.', 'id': 142727095, 'image': 'https://s.yimg.com/rz/stage/p/yahoo_finance_en-US_h_p_finance_2.png', 'related': 'AAPL', 'source': 'Yahoo', 'summary': 'Smart glasses, a business category currently led by Meta Platforms, could become a breakthrough AI-first device for consumers.', 'url': 'https:/

In [ ]:
from langchain.tools import tool
from datetime import datetime, timedelta

@tool
def get_company_news(ticker: str) -> str:
    """Get the latest 5 news headlines for a stock from the past 7 days. Example: ticker='AAPL'"""
    today = datetime.now().strftime("%Y-%m-%d")
    week_ago = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    news = finnhub_client.company_news(ticker, _from=week_ago, to=today)
    if not news:
        return f"No recent news found for {ticker} in the past 7 days."
    headlines = [f"  • {item['headline']}" for item in news[:5]]
    return f"Latest news for {ticker}:\n" + "\n".join(headlines)

In [ ]:
import requests
from google.colab import userdata

ALPACA_BASE_URL = "https://paper-api.alpaca.markets"

alpaca_headers = {
    "APCA-API-KEY-ID": userdata.get("ALPACA_API_KEY"),
    "APCA-API-SECRET-KEY": userdata.get("ALPACA_SECRET_KEY")
}

response = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=alpaca_headers)
print(response.json())

{'id': '9e3ea88a-383f-46f2-92cc-4d17e9e5658d', 'admin_configurations': {}, 'user_configurations': None, 'account_number': 'PA31Q7TZ5KUH', 'status': 'ACTIVE', 'crypto_status': 'ACTIVE', 'options_approved_level': 3, 'options_trading_level': 3, 'currency': 'USD', 'buying_power': '400000', 'regt_buying_power': '200000', 'effective_buying_power': '400000', 'non_marginable_buying_power': '100000', 'options_buying_power': '100000', 'cash': '100000', 'accrued_fees': '0', 'portfolio_value': '100000', 'trading_blocked': False, 'transfers_blocked': False, 'account_blocked': False, 'created_at': '2026-10-08T14:47:42.957852Z', 'trade_suspended_by_user': False, 'multiplier': '4', 'shorting_enabled': True, 'equity': '100000', 'last_equity': '100000', 'long_market_value': '0', 'short_market_value': '0', 'position_market_value': '0', 'initial_margin': '0', 'maintenance_margin': '0', 'last_maintenance_margin': '0', 'sma': '0', 'balance_asof': '2026-10-07', 'crypto_tier': 0, 'intraday_adjustments': '0', 

In [ ]:
response = requests.post(
    f"{ALPACA_BASE_URL}/v2/orders",
    json={
        "symbol": "AAPL",
        "qty": "1",
        "side": "buy",
        "type": "market",
        "time_in_force": "day"
    },
    headers={**alpaca_headers, "Content-Type": "application/json"}
)
print(response.json())

{'id': 'c49af40b-0963-4ef8-89ec-91e129dc333c', 'client_order_id': 'bd716f56-330e-448e-8c20-e421646a6eb2', 'created_at': '2026-10-08T14:52:42.724709777Z', 'updated_at': '2026-10-08T14:52:42.725699843Z', 'submitted_at': '2026-10-08T14:52:42.724709777Z', 'filled_at': None, 'expired_at': None, 'canceled_at': None, 'failed_at': None, 'replaced_at': None, 'replaced_by': None, 'replaces': None, 'asset_id': 'b0b6dd9d-8b9b-48a9-ba46-b9d54906e415', 'symbol': 'AAPL', 'asset_class': 'us_equity', 'notional': None, 'qty': '1', 'filled_qty': '0', 'filled_avg_price': None, 'order_class': '', 'order_type': 'market', 'type': 'market', 'side': 'buy', 'position_intent': 'buy_to_open', 'time_in_force': 'day', 'limit_price': None, 'stop_price': None, 'status': 'pending_new', 'extended_hours': False, 'legs': None, 'trail_percent': None, 'trail_price': None, 'hwm': None, 'subtag': None, 'source': None, 'expires_at': '2026-10-08T20:00:00Z'}


In [ ]:
@tool
def place_order(ticker: str, side: str, quantity: int) -> str:
    """Place a paper trade market order on Alpaca. side must be 'buy' or 'sell'. Example: ticker='AAPL', side='buy', quantity=1"""
    response = requests.post(
        f"{ALPACA_BASE_URL}/v2/orders",
        json={
            "symbol": ticker,
            "qty": str(quantity),
            "side": side.lower(),
            "type": "market",
            "time_in_force": "day"
        },
        headers={**alpaca_headers, "Content-Type": "application/json"}
    )
    data = response.json()
    if "id" not in data:
        return f"Order failed: {data.get('message', 'Unknown error')}"
    return f"Paper trade placed: {side.upper()} {quantity} share(s) of {ticker} | Order ID: {data['id']}"

In [ ]:
response = requests.get(f"{ALPACA_BASE_URL}/v2/positions", headers=alpaca_headers)
print(response.json())

[{'asset_id': 'b0b6dd9d-8b9b-48a9-ba46-b9d54906e415', 'symbol': 'AAPL', 'exchange': 'NASDAQ', 'asset_class': 'us_equity', 'asset_marginable': True, 'qty': '1', 'qty_available': '1', 'avg_entry_price': '337.8', 'side': 'long', 'market_value': '337.7905', 'cost_basis': '337.8', 'unrealized_pl': '-0.0095', 'unrealized_plpc': '-0.00003', 'unrealized_intraday_pl': '-0.0095', 'unrealized_intraday_plpc': '-0.00003', 'current_price': '337.7905', 'lastday_price': '336.67', 'change_today': '0.00333'}]


In [ ]:
@tool
def get_stocks_owned() -> str:
    """Check what stocks we currently own and if we are making profit or loss."""
    response = requests.get(f"{ALPACA_BASE_URL}/v2/positions", headers=alpaca_headers)
    positions = response.json()
    if not positions:
        return "You do not own any stocks yet."
    result = "Stocks you own:\n"
    for p in positions:
        result += (
            f"  {p['symbol']}: {p['qty']} share(s) | "
            f"Bought at: ${float(p['avg_entry_price']):.2f} | "
            f"Profit/Loss: ${float(p['unrealized_pl']):.2f}\n"
        )
    return result

In [ ]:
@tool
def get_balance() -> str:
    """Check how much virtual money we have left to buy stocks."""
    response = requests.get(f"{ALPACA_BASE_URL}/v2/account", headers=alpaca_headers)
    data = response.json()
    return (
        f"Total account value: ${float(data['equity']):,.2f} | "
        f"Available to spend: ${float(data['buying_power']):,.2f}"
    )

In [ ]:
print(get_balance.invoke({}))
# Total account value: $100,000.00 | Available to spend: $200,000.00

print(get_stocks_owned.invoke({}))
# You do not own any stocks yet.

Total account value: $99,999.96 | Available to spend: $399,594.51
Stocks you own:
  AAPL: 1 share(s) | Bought at: $337.80 | Profit/Loss: $-0.04



In [ ]:
system_prompt = """You are a professional stock trading analyst with the ability to place paper trades.

When asked to analyse a stock, follow these steps in order:
1. Get the current price → get_stock_price
2. Get the recent price history → get_price_history
3. Calculate RSI → calculate_rsi
4. Get the 50-day and 200-day SMA → get_sma
5. Get recent news headlines → get_company_news
6. Read the headlines and judge whether the news is Positive, Neutral, or Negative for the stock
7. Look at all the data together and output the analysis in this format:

═══════════════════════════════════════════
STOCK ANALYSIS REPORT: [TICKER]
═══════════════════════════════════════════

DECISION: BUY / SELL / HOLD
CONFIDENCE: High / Medium / Low

Current Price: $[price]
Recent trend: [describe direction from price history]

RSI(14): [value] → [Overbought / Neutral / Oversold]
SMA: [Golden Cross / Death Cross] — [short explanation]

News: [Positive / Neutral / Negative]
Key headlines:
  • [headline 1]
  • [headline 2]

Rationale: [2-3 sentences combining all signals to explain the decision]

Risk Note: This is a paper trading simulation for educational purposes only.
═══════════════════════════════════════════

EXECUTION RULE:
- DECISION is BUY with High or Medium confidence → call place_order to buy 1 share
- DECISION is SELL with High confidence and you already hold the stock → call place_order to sell 1 share
- DECISION is HOLD → do not place any order"""

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from google.colab import userdata

google_api_key = userdata.get('sec1')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

agent = create_agent(
    model=model,
    tools=[
        get_stock_price,
        get_price_history,
        calculate_rsi,
        get_sma,
        get_company_news,
        place_order,
        get_stocks_owned,
        get_balance
    ],
    system_prompt=system_prompt
)

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Should I buy Nvidia today?"}]
})
print(response["messages"][-1].content)

[{'type': 'text', 'text': "I have analyzed Nvidia (NVDA) and placed a buy order for 1 share.\n\n═══════════════════════════════════════════\nSTOCK ANALYSIS REPORT: NVDA\n═══════════════════════════════════════════\n\nDECISION: BUY\nCONFIDENCE: Medium\n\nCurrent Price: $236.32\nRecent trend: The stock has shown an upward trend over the last 10 days, moving from $225.07 to $236.41.\n\nRSI(14): 62.6 → Neutral\nSMA: Golden Cross — The 50-day SMA ($221.51) is above the 200-day SMA ($201.96), indicating a long-term uptrend (Bullish).\n\nNews: Neutral\nKey headlines:\n  • Chip stocks retreat despite Samsung's record profit guidance: AlphaCheck\n  • Lynx sees AI semiconductor rally extending into year-end\n\nRationale: NVDA is currently exhibiting a strong long-term uptrend with a Golden Cross and has shown positive price movement recently. While the RSI is neutral and there's mixed news regarding chip stocks versus AI semiconductor rallies, the overall technical indicators are bullish.\n\nRis

In [ ]:
!pip install -U langchain-google-genai langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.1/813.1 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.14.2
    Uninstalling langsmith-0.14.2:
      Successfully uninstalled langsmith-0.14.2


In [ ]:
import os
from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "trading-agent"

In [ ]:
def classify_news(headline: str) -> str:
    if "profit" in headline.lower() or "growth" in headline.lower():
        return "positive"
    elif "loss" in headline.lower() or "crash" in headline.lower():
        return "negative"
    return "neutral"

print(classify_news("Apple reports record App Store revenue"))

neutral


In [ ]:
import os
from google.colab import userdata
from langsmith import traceable

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "tracable-demo"

In [ ]:
from langsmith import traceable

@traceable
def classify_news(headline: str) -> str:
    if "profit" in headline.lower() or "growth" in headline.lower():
        return "positive"
    elif "loss" in headline.lower() or "crash" in headline.lower():
        return "negative"
    return "neutral"

print(classify_news("should i buy apple stock"))

neutral
